In [ ]:
import os
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from PIL import Image
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [ ]:
#If using Google Colab and storing the dataset on Google Drive
#from google.colab import drive
#drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Handling images (stored in 3 folders, as organized does when downloading the data)
image_root_path = #image path
image_path_mapping = {}
for root, dirs, files in os.walk(image_root_path):
  for file in files:
      if file.endswith(".png"):
          image_path_mapping[file] = os.path.join(root, file)

print(f"Indexing complete: {len(image_path_mapping)} PNG images found.")

Indexing complete: 2298 PNG images found.


In [ ]:
# 1. Configuration & Hyperparameters
CONFIG = {
    "image_size": 224,
    "batch_size": 32,
    "lr": 0.0001,
    "epochs": 20,
    "n_splits": 5,  # For K-Fold Cross Validation
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

In [ ]:
# 2. Data Preprocessing (Metadata)
def preprocess_metadata(df):
    df = df.copy()

    # Fill missing values for clinical data
    df['age'] = df['age'].fillna(df['age'].mean())
    df['smoke'] = df['smoke'].fillna(df['smoke'].mode()[0])
    df['drink'] = df['drink'].fillna(df['drink'].mode()[0])

    # Identify ALL non-numeric columns to encode them
    categorical_cols = df.select_dtypes(include=['object']).columns

    le = LabelEncoder()
    for col in categorical_cols:
        if col not in ['patient_id', 'lesion_id', 'img_id', 'diagnostic']:
            df[col] = le.fit_transform(df[col].astype(str))

    # Scale numerical data (Age) - Removed to prevent data leakage
    #scaler = StandardScaler()
    #df[['age']] = scaler.fit_transform(df[['age']])

    # Final safety check: fill any remaining NaNs with 0
    df = df.fillna(0)

    return df

In [ ]:
# 3. Custom Multimodal Dataset
class SkinCancerDataset(Dataset):
    def __init__(self, df, path_mapping, label_mapping, transform=None):
        self.df = df
        self.path_mapping = path_mapping
        self.transform = transform
        self.label_mapping = label_mapping

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['img_id']

        if not img_id.endswith(".png"):
            img_id += ".png"

        img_path = self.path_mapping.get(img_id)

        if img_path is None:
            raise FileNotFoundError(f"Image {img_id} not found.")

        image = Image.open(img_path).convert('RGB')

        metadata = self.df.iloc[idx].drop(['patient_id', 'lesion_id', 'img_id', 'diagnostic']).values.astype(np.float32)
        label = self.label_mapping[self.df.iloc[idx]['diagnostic']]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(metadata), torch.tensor(label)

In [ ]:
# 4. Multimodal Fusion Model (CNN + MLP)
class MultimodalNet(nn.Module):
    def __init__(self, num_classes, num_metadata_features):
        super(MultimodalNet, self).__init__()

        # Image Branch (ResNet50)
        self.cnn = resnet50(weights=ResNet50_Weights.DEFAULT)
        num_ftrs = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity() # Remove final FC layer

        # Metadata Branch (MLP)
        self.mlp = nn.Sequential(
            nn.Linear(num_metadata_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Fusion Layer
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 32, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, img, meta):
        img_features = self.cnn(img) # Shape: [batch, 2048]
        meta_features = self.mlp(meta) # Shape: [batch, 32]

        combined = torch.cat((img_features, meta_features), dim=1)
        output = self.classifier(combined)
        return output


In [ ]:
# 5. Training and Evaluation Pipeline
def train_model(model, train_loader, criterion, optimizer):
    model.to(CONFIG['device'])

    for epoch in range(CONFIG['epochs']):
        model.train()
        running_loss = 0.0

        for images, metadata, labels in train_loader:
            images, metadata, labels = images.to(CONFIG['device']), metadata.to(CONFIG['device']), labels.to(CONFIG['device'])

            optimizer.zero_grad()
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{CONFIG['epochs']}, Loss: {running_loss/len(train_loader):.4f}")



In [ ]:
#6. Feature importance
from sklearn.inspection import permutation_importance

def analyze_feature_importance(model, feature_names):
    model.eval()

    with torch.no_grad():
        weights = model.mlp[0].weight.data.abs().mean(dim=0).cpu().numpy()

    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': weights
    })

    # Normalize weights so they sum to 1.0
    feature_importance_df['Importance'] /= feature_importance_df['Importance'].sum()

    return feature_importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
# 7. Main Execution Script
if __name__ == "__main__":

    # Loading the metadata
    path = #metadata path

    try:
        df = pd.read_csv(path)
        df = df.drop_duplicates(keep='first')
        print("Dataframe loaded. Shape:", df.shape)
    except:
        print("Check if metadata.csv is in the folder")

    df = preprocess_metadata(df)


    # Image Transformations
    transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    global_label_mapping = {label: i for i, label in enumerate(sorted(df['diagnostic'].unique()))}
    print("Global Class Mapping Reference:", global_label_mapping)

    meta_features_names = df.drop(['patient_id', 'lesion_id', 'img_id', 'diagnostic'], axis=1).columns.tolist()
    global_feature_importances = []

    # K-Fold Cross Validation
    skf = StratifiedKFold(n_splits=CONFIG['n_splits'])

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['diagnostic'])):
        print(f"--- Training Fold {fold+1} ---")

        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()

        scaler = StandardScaler()
        train_df[['age']] = scaler.fit_transform(train_df[['age']])
        val_df[['age']] = scaler.transform(val_df[['age']])

        train_ds = SkinCancerDataset(train_df, image_path_mapping, global_label_mapping, transform=transform)
        val_ds = SkinCancerDataset(val_df, image_path_mapping, global_label_mapping, transform=transform)

        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)

        num_meta = train_df.drop(['patient_id', 'lesion_id', 'img_id', 'diagnostic'], axis=1).shape[1]
        model = MultimodalNet(num_classes=len(df['diagnostic'].unique()), num_metadata_features=num_meta)

        optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])

        train_class_counts = train_df['diagnostic'].value_counts()
        ordered_counts = [train_class_counts.get(name, 1) for name in global_label_mapping.keys()]
        weights = 1. / torch.tensor(ordered_counts, dtype=torch.float)
        weights = weights.to(CONFIG['device'])

        criterion = nn.CrossEntropyLoss(weight=weights)

        train_model(model, train_loader, criterion, optimizer)

        # Feature Importance
        fold_importance = analyze_feature_importance(model, meta_features_names)
        global_feature_importances.append(fold_importance)

        # --- Evaluation Phase ---
        model.eval()
        y_true = []
        y_pred = []

        with torch.no_grad():
            for imgs, metas, labels in val_loader:
                imgs, metas = imgs.to(CONFIG['device']), metas.to(CONFIG['device'])
                outputs = model(imgs, metas)
                preds = torch.argmax(outputs, dim=1)

                y_true.extend(labels.numpy())
                y_pred.extend(preds.cpu().numpy())

        # Generation of the Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)

        # Calculation of Specificity and Sensitivity for each class
        print(f"\nResults for Fold {fold+1}:")

        for class_name, idx in global_label_mapping.items():
          tp = cm[idx, idx] if idx < cm.shape[0] else 0
          fp = cm[:, idx].sum() - tp if idx < cm.shape[1] else 0
          fn = cm[idx, :].sum() - tp if idx < cm.shape[0] else 0
          tn = cm.sum() - (tp + fp + fn)

          sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
          specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

          print(f"Class [{class_name}]:")
          print(f"  - Sensitivity (Recall): {sensitivity:.4f}")
          print(f"  - Specificity: {specificity:.4f}")

        # Force the classification report to follow the global dictionary ordering
        print("\nFull Classification Report:")
        print(classification_report(y_true, y_pred, target_names=list(global_label_mapping.keys()), zero_division=0))

    print("----------------------------- ")
    print("Global Feature Importance  ")
    master_importance = pd.concat(global_feature_importances).groupby('Feature').mean().reset_index()
    master_importance = master_importance.sort_values(by='Importance', ascending=False).reset_index(drop=True)
    print(master_importance)